In [4]:
!pip install -q git+https://github.com/RayenSansa03/aeroguard.git@main --force-reinstall --no-deps
!nvidia-smi --query-gpu=name --format=csv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
name
Tesla T4


In [12]:
from google.colab import drive
drive.mount("/content/drive")

import json
import time
import numpy as np
import pandas as pd
import xgboost
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from aeroguard.donnees_ml import preparer_xy, preparer_xy_famille

DOSSIER = "/content/drive/MyDrive/aeroguard"
vols = pd.read_parquet(f"{DOSSIER}/data/processed/vols_ncmapss.parquet")
X_train, y_train = preparer_xy(vols, "train")

# Configuration figée : choisie sur la validation (leçons 2.7, 3.2, 3.3, 3.4),
# elle ne doit plus changer une fois le jeu de test évalué.
CONFIG_FIGEE = {
    "logistique": {"seuil": 0.50, "k": 3},
    "xgboost": {"seuil": 0.81, "k": 3, "fichier": "xgb_binaire_v3.json"},
}
print(json.dumps(CONFIG_FIGEE, indent=2))

# Modèles d'alerte
alerte_logistique = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                  LogisticRegression(max_iter=1000)).fit(X_train, y_train)
alerte_xgboost = xgboost.XGBClassifier()
alerte_xgboost.load_model(f"{DOSSIER}/models/{CONFIG_FIGEE['xgboost']['fichier']}")
alerte_xgboost.set_params(device="cpu")

# Modèles de diagnostic
Xf_train, fam_train = preparer_xy_famille(vols, "train")
encodeur_familles = LabelEncoder().fit(fam_train)
yf_train = encodeur_familles.transform(fam_train)
diagnostic_logistique = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                      LogisticRegression(max_iter=3000))
diagnostic_logistique.fit(
    Xf_train, yf_train,
    logisticregression__sample_weight=compute_sample_weight("balanced", yf_train))
diagnostic_xgboost = xgboost.XGBClassifier()
diagnostic_xgboost.load_model(f"{DOSSIER}/models/xgb_diagnostic_regle_v3.json")
diagnostic_xgboost.set_params(device="cpu")
print("Modèles chargés, configuration figée.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "logistique": {
    "seuil": 0.5,
    "k": 3
  },
  "xgboost": {
    "seuil": 0.81,
    "k": 3,
    "fichier": "xgb_binaire_v3.json"
  }
}
Modèles chargés, configuration figée.


In [13]:
X_test, y_test = preparer_xy(vols, "test")
X_val, y_val = preparer_xy(vols, "val")

valeurs_manquantes = X_test.isna()
print("NaN — train :", int(X_train.isna().sum().sum()), "| validation :", int(X_val.isna().sum().sum()))
print(f"Vols du test avec au moins un NaN : {valeurs_manquantes.any(axis=1).sum()} sur {len(X_test)}")
print("\nFeatures les plus touchées :")
print(valeurs_manquantes.sum().loc[lambda s: s > 0].sort_values(ascending=False).head(10))
print("\nMoteurs concernés :")
print(vols.loc[X_test.index[valeurs_manquantes.any(axis=1)], "moteur"].value_counts())

NaN — train : 0 | validation : 0
Vols du test avec au moins un NaN : 30 sur 2938

Features les plus touchées :
T24_mean_montee     30
T24_std_montee      30
T24_std_descente    30
T24_max_montee      30
T50_mean_montee     30
T30_mean_montee     30
T30_std_descente    30
T30_std_montee      30
T30_max_montee      30
T48_std_descente    30
dtype: int64

Moteurs concernés :
moteur
DS02_14    30
Name: count, dtype: int64


In [14]:
from aeroguard.metier import evaluer_alerte

infos_test = vols.loc[X_test.index, ["moteur", "cycle", "RUL"]]
assert infos_test["moteur"].nunique() == 39, "Le jeu de test doit contenir 39 moteurs."

resultats_test = {}
for nom, modele in [("logistique", alerte_logistique), ("xgboost", alerte_xgboost)]:
    probas = modele.predict_proba(X_test)[:, 1]
    resultats_test[nom] = evaluer_alerte(y_test, probas, CONFIG_FIGEE[nom]["seuil"],
                                         infos_test, k=CONFIG_FIGEE[nom]["k"])

colonnes = ["f1_macro", "pr_auc", "rappel", "precision", "fp", "fn",
            "moteurs", "moteurs_detectes", "avance_moyenne", "avance_mediane",
            "moteurs_avec_fausse_alarme", "fausses_alarmes_total"]
bilan_test_alerte = pd.DataFrame(resultats_test).T[colonnes]
bilan_test_alerte.round(3)

,f1_macro,pr_auc,rappel,precision,fp,fn,moteurs,moteurs_detectes,avance_moyenne,avance_mediane,moteurs_avec_fausse_alarme,fausses_alarmes_total
logistique,0.840,0.978,0.922,0.882,245.0,156.0,39.0,39.0,44.718,43.0,12.0,54.0
xgboost,0.852,0.978,0.863,0.933,123.0,272.0,39.0,39.0,40.051,39.0,4.0,28.0


In [15]:
reference_validation = pd.DataFrame({
    "logistique": {"f1_macro": 0.886, "avance_moyenne": 46.1, "fausses_alarmes_total": 9},
    "xgboost": {"f1_macro": 0.885, "avance_moyenne": 42.5, "fausses_alarmes_total": 2},
}).T
validation_vs_test = pd.concat({
    "validation (11 moteurs)": reference_validation,
    "test (39 moteurs)": bilan_test_alerte[reference_validation.columns],
}, axis=1)

# Fausses alarmes par moteur : comparaison équitable entre 11 et 39 moteurs
validation_vs_test[("validation (11 moteurs)", "fausses_alarmes_par_moteur")] = (
    validation_vs_test[("validation (11 moteurs)", "fausses_alarmes_total")] / 11)
validation_vs_test[("test (39 moteurs)", "fausses_alarmes_par_moteur")] = (
    validation_vs_test[("test (39 moteurs)", "fausses_alarmes_total")].astype(float) / 39)
validation_vs_test.sort_index(axis=1).round(3)

test (39 moteurs)                                      \
              avance_moyenne f1_macro fausses_alarmes_par_moteur   
logistique            44.718    0.840                      1.385   
xgboost               40.051    0.852                      0.718   

                                 validation (11 moteurs)           \
           fausses_alarmes_total          avance_moyenne f1_macro   
logistique                  54.0                    46.1    0.886   
xgboost                     28.0                    42.5    0.885   

                                                             
           fausses_alarmes_par_moteur fausses_alarmes_total  
logistique                      0.818                   9.0  
xgboost                         0.182                   2.0

In [16]:
from sklearn.metrics import accuracy_score, f1_score
from aeroguard.metier import diagnostic_par_moteur

Xf_test, fam_test = preparer_xy_famille(vols, "test")
yf_test = encodeur_familles.transform(fam_test)
moteurs_test = vols.loc[Xf_test.index, "moteur"]
famille_reelle = vols.loc[Xf_test.index].groupby("moteur")["famille"].first()

bilan_test_diagnostic = {}
for nom, modele in [("logistique", diagnostic_logistique), ("xgboost_optuna", diagnostic_xgboost)]:
    predictions = modele.predict(Xf_test)
    diagnostic_moteur = diagnostic_par_moteur(moteurs_test, encodeur_familles.inverse_transform(predictions))
    bilan_test_diagnostic[nom] = {
        "exactitude": accuracy_score(yf_test, predictions),
        "f1_macro": f1_score(yf_test, predictions, average="macro"),
        "moteurs_bien_diagnostiques": int((diagnostic_moteur == famille_reelle).sum()),
        "moteurs": len(famille_reelle),
    }
bilan_test_diagnostic = pd.DataFrame(bilan_test_diagnostic).T
bilan_test_diagnostic.round(3)

,exactitude,f1_macro,moteurs_bien_diagnostiques,moteurs
logistique,0.838,0.846,35.0,39.0
xgboost_optuna,0.822,0.826,37.0,39.0


In [17]:
from aeroguard.modeles import creer_xgboost

with open(f"{DOSSIER}/models/xgb_diagnostic_reglages_v3.json") as f:
    reglages_diagnostic = json.load(f)

familles_train = vols.loc[X_train.index, "famille"].values
familles_test = vols.loc[X_test.index, "famille"].values
resultats_familles = []

for famille in sorted(set(familles_train)):
    # 1) Alerte réentraînée sans cette famille
    masque_entrainement = familles_train != famille
    modele_sans_famille = creer_xgboost("cuda", early_stopping_rounds=None, n_estimators=411)
    modele_sans_famille.fit(X_train[masque_entrainement], y_train[masque_entrainement])
    modele_sans_famille.set_params(device="cpu")

    masque_famille = familles_test == famille
    avec_famille = evaluer_alerte(
        y_test[masque_famille], alerte_xgboost.predict_proba(X_test[masque_famille])[:, 1],
        0.81, infos_test[masque_famille], k=3)
    sans_famille = evaluer_alerte(
        y_test[masque_famille], modele_sans_famille.predict_proba(X_test[masque_famille])[:, 1],
        0.81, infos_test[masque_famille], k=3)

    # 2) Diagnostic réentraîné sans cette famille : quelle famille prédit-il à la place ?
    masque_diag = fam_train.values != famille
    encodeur_reduit = LabelEncoder().fit(fam_train[masque_diag])
    yf_reduit = encodeur_reduit.transform(fam_train[masque_diag])
    diagnostic_sans_famille = creer_xgboost("cuda", objective="multi:softprob",
                                            early_stopping_rounds=None, **reglages_diagnostic)
    diagnostic_sans_famille.fit(Xf_train[masque_diag], yf_reduit,
                                sample_weight=compute_sample_weight("balanced", yf_reduit))
    diagnostic_sans_famille.set_params(device="cpu")
    probas_familles = diagnostic_sans_famille.predict_proba(Xf_test[fam_test.values == famille])
    repartition_predictions = pd.Series(
        encodeur_reduit.inverse_transform(probas_familles.argmax(axis=1))
    ).value_counts(normalize=True)

    resultats_familles.append({
        "famille_non_vue": famille,
        "moteurs_test": avec_famille["moteurs"],
        "rappel_avec": avec_famille["rappel"], "rappel_sans": sans_famille["rappel"],
        "detectes_avec": avec_famille["moteurs_detectes"], "detectes_sans": sans_famille["moteurs_detectes"],
        "avance_avec": avec_famille["avance_moyenne"], "avance_sans": sans_famille["avance_moyenne"],
        "diagnostic_predit": f"{repartition_predictions.index[0]} ({repartition_predictions.iloc[0]:.0%})",
        "confiance_moyenne": probas_familles.max(axis=1).mean(),
    })

bilan_famille_non_vue = pd.DataFrame(resultats_familles).set_index("famille_non_vue")
bilan_famille_non_vue.round(2)

,moteurs_test,rappel_avec,rappel_sans,detectes_avec,detectes_sans,avance_avec,avance_sans,diagnostic_predit,confiance_moyenne
famille_non_vue,,,,,,,,,
HPC,4,0.88,0.84,4,4,43.00,41.50,LPC+HPC (92%),0.93
HPT,4,0.90,0.81,4,4,46.50,39.75,HPT+LPT (43%),0.78
HPT+LPT,9,0.95,0.96,9,9,40.89,42.33,LPT (68%),0.89
LPC+HPC,4,0.83,0.82,4,4,42.50,42.50,HPC (86%),0.89
LPT,4,0.81,0.74,4,4,40.50,38.25,HPT+LPT (92%),0.90
fan,4,0.78,0.22,4,2,46.50,24.50,mixte (88%),0.92
mixte,10,0.85,0.91,10,10,31.80,35.20,LPC+HPC (45%),0.73


In [18]:
rng = np.random.default_rng(42)
X_charge = pd.concat([X_train] * 100, ignore_index=True)
X_charge = X_charge + rng.normal(0, 0.01, X_charge.shape).astype("float32")
y_charge = np.tile(y_train.values, 100)
print(f"Jeu de charge : {len(X_charge):,} vols × {X_charge.shape[1]} features")

mesures = []
for device in ["cuda", "cpu"]:
    modele = creer_xgboost(device, early_stopping_rounds=None, n_estimators=300)
    debut = time.perf_counter()
    modele.fit(X_charge, y_charge)
    duree_entrainement = time.perf_counter() - debut
    debut = time.perf_counter()
    modele.predict_proba(X_charge)
    duree_prediction = time.perf_counter() - debut
    mesures.append({"device": device, "entrainement_s": duree_entrainement,
                    "prediction_s": duree_prediction})

bilan_vitesse_gpu = pd.DataFrame(mesures).set_index("device")
bilan_vitesse_gpu.loc["acceleration (cpu / gpu)"] = bilan_vitesse_gpu.loc["cpu"] / bilan_vitesse_gpu.loc["cuda"]
bilan_vitesse_gpu.round(2)

Jeu de charge : 371,600 vols × 126 features


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [20:47:07] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,entrainement_s,prediction_s
device,,
cuda,12.41,1.74
cpu,52.83,1.47
acceleration (cpu / gpu),4.26,0.85


In [19]:
import os

os.makedirs(f"{DOSSIER}/results", exist_ok=True)
bilan_test_alerte.to_csv(f"{DOSSIER}/results/v3_test_alerte.csv")
bilan_test_diagnostic.to_csv(f"{DOSSIER}/results/v3_test_diagnostic.csv")
bilan_famille_non_vue.to_csv(f"{DOSSIER}/results/v3_famille_non_vue.csv")
bilan_vitesse_gpu.to_csv(f"{DOSSIER}/results/v3_vitesse_gpu.csv")
print("Résultats de la v3 sauvegardés.")

Résultats de la v3 sauvegardés.
